In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from yellowbrick.target import ClassBalance
from yellowbrick.features import (
    # HistogramVisualizer,
    Rank1D, Rank2D, JointPlotVisualizer,
    ParallelCoordinates, RadViz, Manifold,
)
from yellowbrick.target import BalancedBinningReference, FeatureCorrelation

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

In [ ]:
df = pd.read_csv("../assignments/DATA_OFFER_ALLOCATION.csv")
df["SESSION_DATE"] = pd.to_datetime(df["SESSION_DATE"])
print(df.shape)
df.head(3)

## 1 — Data Overview & Preprocessing

In [ ]:
# Sentinel values: 9999 = "never / no history", -1 = "no transaction context"
# We keep them as-is for most Yellowbrick plots (they will show as a visible spike)
# but also create a clean version where sentinels → NaN for correlation analysis.

SENTINEL_9999 = ["DAYS_SINCE_LAST_PURCHASE_L12M", "DAYS_SINCE_LAST_VISIT_NO_PURCHASE"]
SENTINEL_NEG1 = [
    "OFFER_RICHNESS_APPLIED", "PRICE_PER_POINT",
    "LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M",
    "OFFER_RICHNESS_APPLIED_ON_LAST_PURCHASE_L12M",
]

df_clean = df.copy()
for col in SENTINEL_9999:
    df_clean[col] = df_clean[col].replace(9999, np.nan)
for col in SENTINEL_NEG1:
    df_clean[col] = df_clean[col].replace(-1, np.nan)

# Feature matrix (numeric predictors only, excluding keys / dates / post-treatment columns)
FEATURE_COLS = [
    "FLAG_FIRST_TIME_VISITOR", "FLAG_FIRST_TIME_BUYER",
    "OFFER_RICHNESS_SERVED",
    "CURRENT_BALANCE",
    "DAYS_SINCE_LAST_PURCHASE_L12M", "COUNT_TRANX_L12M",
    "LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M",
    "OFFER_RICHNESS_APPLIED_ON_LAST_PURCHASE_L12M",
    "POINTS_PURCHASED_LAST_TRANX_L12M",
    "DAYS_SINCE_LAST_VISIT_NO_PURCHASE",
]
TARGET = "FLAG_TRANSACTION"

X = df_clean[FEATURE_COLS].fillna(-1).values
y = df_clean[TARGET].values
feature_names = FEATURE_COLS

print("Missing values (clean df):\n", df_clean[FEATURE_COLS].isna().sum())
print("\nClass counts:\n", df_clean[TARGET].value_counts())

## 2 — Target Distribution (Class Balance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall class balance
viz = ClassBalance(labels=["No Transaction", "Transaction"], ax=axes[0])
viz.fit(y)
viz.finalize()
axes[0].set_title("Overall class balance — FLAG_TRANSACTION")

# Class balance by offer richness level
offer_levels = sorted(df["OFFER_RICHNESS_SERVED"].unique())
conv_rates = [
    df[df["OFFER_RICHNESS_SERVED"] == lvl]["FLAG_TRANSACTION"].mean()
    for lvl in offer_levels
]
axes[1].bar([str(l) for l in offer_levels], conv_rates, color="steelblue", edgecolor="white")
axes[1].set_xlabel("OFFER_RICHNESS_SERVED")
axes[1].set_ylabel("Conversion rate")
axes[1].set_title("Conversion rate by offer richness")
for i, v in enumerate(conv_rates):
    axes[1].text(i, v + 0.002, f"{v:.1%}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 3 — Feature Distributions (HistogramVisualizer)

In [ ]:
# Continuous features worth histogramming (exclude binary flags & sentinel-heavy cols)
HIST_COLS = [
    "CURRENT_BALANCE", "POINTS_PURCHASED_LAST_TRANX_L12M",
    "DAYS_SINCE_LAST_PURCHASE_L12M", "DAYS_SINCE_LAST_VISIT_NO_PURCHASE",
    "COUNT_TRANX_L12M",
]

from yellowbrick.target import BalancedBinningReference

visualizer = BalancedBinningReference()

visualizer.fit(df[HIST_COLS[0]])
visualizer.show()

# fig, axes = plt.subplots(1, len(HIST_COLS), figsize=(18, 4))
# for ax, col in zip(axes, HIST_COLS):
#     viz = HistogramVisualizer(feature=col, ax=ax, bins=30, color="steelblue")
#     viz.fit_transform(df_clean[[col]].dropna().values)
#     viz.finalize()
#     ax.set_title(col, fontsize=9)

# plt.suptitle("Feature distributions (sentinels removed)", y=1.02, fontsize=12)
# plt.tight_layout()
# plt.show()

## 4 — Feature Rankings: Rank1D & Rank2D (Pearson correlations)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Rank1D — Shapiro ranking of each feature individually
viz1 = Rank1D(features=feature_names, algorithm="shapiro", ax=axes[0])
viz1.fit_transform(X, y)
viz1.finalize()
axes[0].set_title("Rank1D — Shapiro ranking")

# Rank2D — Pearson pairwise correlation heatmap
viz2 = Rank2D(features=feature_names, algorithm="pearson", ax=axes[1])
viz2.fit_transform(X, y)
viz2.finalize()
axes[1].set_title("Rank2D — Pearson pairwise correlation")

plt.tight_layout()
plt.show()

## 5 — Feature–Target Correlation (FeatureCorrelation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Pearson correlation of each feature vs target
viz_p = FeatureCorrelation(method="pearson", features=feature_names, ax=axes[0])
viz_p.fit(X, y)
viz_p.finalize()
axes[0].set_title("Pearson correlation with FLAG_TRANSACTION")

# Mutual information (handles nonlinear relationships)
viz_mi = FeatureCorrelation(method="mutual_info-classification", features=feature_names, ax=axes[1])
viz_mi.fit(X, y)
viz_mi.finalize()
axes[1].set_title("Mutual information with FLAG_TRANSACTION")

plt.tight_layout()
plt.show()

In [ ]:
feature_names

## 6 — Confounders: Are customer features predictive of which offer they receive?

If `OFFER_RICHNESS_SERVED` is correlated with pre-treatment covariates, the data is observational (not a clean RCT) and naive uplift estimates will be biased.

In [ ]:
COVARIATE_COLS = [
    "FLAG_FIRST_TIME_VISITOR", "FLAG_FIRST_TIME_BUYER",
    "CURRENT_BALANCE", "COUNT_TRANX_L12M",
    "DAYS_SINCE_LAST_PURCHASE_L12M", "DAYS_SINCE_LAST_VISIT_NO_PURCHASE",
]

X_cov = df_clean[COVARIATE_COLS].fillna(-1).values
y_offer = df_clean["OFFER_RICHNESS_SERVED"].values  # treat offer as "target" for this check

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Pearson correlation between covariates and the OFFER assigned
viz_conf = FeatureCorrelation(method="pearson", features=COVARIATE_COLS, ax=axes[0])
viz_conf.fit(X_cov, y_offer)
viz_conf.finalize()
axes[0].set_title("Pearson corr: covariates → OFFER_RICHNESS_SERVED\n(non-zero = potential confounder)")

# Rank2D on covariates only — see which customer features co-vary (multicollinearity)
viz_cov2 = Rank2D(features=COVARIATE_COLS, algorithm="pearson", ax=axes[1])
viz_cov2.fit_transform(X_cov, y_offer)
viz_cov2.finalize()
axes[1].set_title("Rank2D — covariate intercorrelations")

plt.tight_layout()
plt.show()

In [ ]:
# Box plots: distribution of key covariates per offer richness level
# A well-randomised trial should show flat medians across offer levels.
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, COVARIATE_COLS):
    groups = [
        df_clean[df_clean["OFFER_RICHNESS_SERVED"] == lvl][col].dropna()
        for lvl in offer_levels
    ]
    ax.boxplot(groups, labels=[str(l) for l in offer_levels], showfliers=False)
    ax.set_xlabel("OFFER_RICHNESS_SERVED")
    ax.set_title(col, fontsize=9)
plt.suptitle("Covariate balance across offer richness levels\n(flat ≈ good randomisation, skewed ≈ confounder)", y=1.01)
plt.tight_layout()
plt.show()

## 7 — Parallel Coordinates (multi-dimensional patterns by outcome)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Subsample for readability — 800 rows, balanced 50/50
rng = np.random.default_rng(42)
idx0 = rng.choice(np.where(y == 0)[0], 400, replace=False)
idx1 = rng.choice(np.where(y == 1)[0], 400, replace=False)
idx = np.concatenate([idx0, idx1])

PC_COLS = [
    "FLAG_FIRST_TIME_BUYER", "OFFER_RICHNESS_SERVED",
    "CURRENT_BALANCE", "COUNT_TRANX_L12M",
    "DAYS_SINCE_LAST_PURCHASE_L12M",
]
X_pc = MinMaxScaler().fit_transform(df_clean[PC_COLS].fillna(-1).values)
y_pc = y[idx]
X_pc = X_pc[idx]

fig, ax = plt.subplots(figsize=(14, 5))
viz = ParallelCoordinates(
    classes=["No Transaction", "Transaction"],
    features=PC_COLS,
    alpha=0.05,
    ax=ax,
)
viz.fit_transform(X_pc, y_pc)
viz.finalize()
ax.set_title("Parallel Coordinates — buyer vs non-buyer patterns (subsampled, balanced)")
plt.tight_layout()
plt.show()

## 8 — RadViz (radial visualisation of feature space by class)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
viz = RadViz(
    classes=["No Transaction", "Transaction"],
    features=PC_COLS,
    alpha=0.3,
    ax=ax,
)
viz.fit_transform(X_pc, y_pc)
viz.finalize()
ax.set_title("RadViz — feature space projection by transaction outcome")
plt.tight_layout()
plt.show()

## 9 — Joint Plots (bivariate relationships)

In [ ]:
pairs = [
    ("CURRENT_BALANCE", "OFFER_RICHNESS_SERVED"),
    ("COUNT_TRANX_L12M", "OFFER_RICHNESS_SERVED"),
    ("CURRENT_BALANCE", "COUNT_TRANX_L12M"),
]

for x_feat, y_feat in pairs:
    fig, ax = plt.subplots(figsize=(6, 6))
    data = df_clean[[x_feat, y_feat]].dropna()
    viz = JointPlotVisualizer(feature=x_feat, target=y_feat, ax=ax)
    viz.fit_transform(data[[x_feat]].values, data[y_feat].values)
    viz.finalize()
    ax.set_title(f"{x_feat}  vs  {y_feat}")
    plt.tight_layout()
    plt.show()

## 10 — Manifold (t-SNE) — latent cluster structure

In [ ]:
# t-SNE on a 1 000-row balanced subsample (t-SNE is O(n²) — keep it small)
n_tsne = 500
idx0_t = rng.choice(np.where(y == 0)[0], n_tsne, replace=False)
idx1_t = rng.choice(np.where(y == 1)[0], n_tsne, replace=False)
idx_t = np.concatenate([idx0_t, idx1_t])

X_tsne = MinMaxScaler().fit_transform(
    df_clean[COVARIATE_COLS + ["OFFER_RICHNESS_SERVED"]].fillna(-1).values
)[idx_t]
y_tsne = y[idx_t]

fig, ax = plt.subplots(figsize=(8, 7))
viz = Manifold(
    manifold="tsne",
    classes=["No Transaction", "Transaction"],
    random_state=42,
    ax=ax,
)
viz.fit_transform(X_tsne, y_tsne)
viz.finalize()
ax.set_title("t-SNE manifold — transaction outcomes in feature space\n(n=1 000, balanced)")
plt.tight_layout()
plt.show()

---
## EDA checklist

| Section | What to look for |
|---|---|
| **2 — Class balance** | Is FLAG_TRANSACTION heavily imbalanced? Does conversion rate rise monotonically with offer richness? |
| **3 — Distributions** | Skew in CURRENT_BALANCE / POINTS_PURCHASED? Heavy tails suggest log-transform before modelling. |
| **4 — Rank2D** | High Pearson correlation between two features → consider dropping one (multicollinearity). |
| **5 — Feature–target corr** | Which features have highest mutual information with the transaction outcome? Prioritise these. |
| **6 — Confounders** | Non-zero Pearson corr between covariates and OFFER_RICHNESS_SERVED → data is observational → use propensity-score weighting or causal inference methods. |
| **7 — Parallel Coords** | Do buyers cluster on specific feature combinations? Are there "easy" vs "hard" converters? |
| **8 — RadViz** | Separation of classes → are the features sufficient to discriminate outcomes? |
| **9 — Joint Plots** | Non-linear relationships between balance/transaction history and offer richness? |
| **10 — t-SNE** | Mixed classes = hard problem. Clean clusters = latent segments to model separately. |